# HCC1806: Feature selection + classifiers (DropSeq)

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.feature_selection import VarianceThreshold, mutual_info_classif
from sklearn.metrics import classification_report, ConfusionMatrixDisplay
import pickle
import os

os.makedirs('../outputs/models', exist_ok=True)
os.makedirs('../outputs/figures', exist_ok=True)

## 1. Load data

In [4]:
train_raw = pd.read_csv('../data/DropSeq/HCC1806_Filtered_Normalised_3000_Data_train.txt',
                        sep=' ', index_col=0)
test_raw  = pd.read_csv('../data/DropSeq/HCC1806_Filtered_Normalised_3000_Data_test_anonim.txt',
                        sep=' ', index_col=0)

train_raw = train_raw.T
test_raw  = test_raw.T

print('Train shape (cells x genes):', train_raw.shape)
print('Test shape (cells x genes):', test_raw.shape)
print('Train sample cell names:', train_raw.index[:3].tolist())
print('Test sample cell names:', test_raw.index[:3].tolist())

Train shape (cells x genes): (14682, 3000)
Test shape (cells x genes): (3671, 3000)
Train sample cell names: ['AAAAAACCCGGC_Normoxia', 'AAAACCGGATGC_Normoxia', 'AAAACGAGCTAG_Normoxia']
Test sample cell names: ['1', '2', '3']


## 2. Extract labels

In [7]:
y_train = train_raw.index.str.split('_').str[-1].values

x_train = train_raw.values
x_test  = test_raw.values

print('Train label distribution:', pd.Series(y_train).value_counts().to_dict())
print('Test cells (no labels):', x_test.shape[0])

Train label distribution: {'Hypoxia': 8899, 'Normoxia': 5783}
Test cells (no labels): 3671


## 3. Feature selection

In [8]:
var_filter = VarianceThreshold(threshold=0.01)
x_train_var = var_filter.fit_transform(x_train)
x_test_var  = var_filter.transform(x_test)

print(f'Genes before variance filter: {x_train.shape[1]}')
print(f'Genes after variance filter: {x_train_var.shape[1]}')

Genes before variance filter: 3000
Genes after variance filter: 1113


In [ ]:
N_TOP_GENES = 500

mi_scores = mutual_info_classif(x_train_var, y_train, random_state=42)
top_idx   = np.argsort(mi_scores)[::-1][:N_TOP_GENES]

x_train_top = x_train_var[:, top_idx]
x_test_top  = x_test_var[:, top_idx]

print(f'Genes after mutual information selection: {x_train_top.shape[1]}')

genes_after_var = train_raw.columns[var_filter.get_support()]
top_genes = genes_after_var[top_idx]
pd.Series(mi_scores[top_idx], index=top_genes).sort_values(ascending=False).to_csv('../outputs/HCC1806_top_genes.csv')